# 1. Library calling

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime
import warnings
import numpy as np
from IPython.display import clear_output
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

from janitor import xlsx_table

# 2. Defining the Product Information and Location

In [2]:
HSN="8716"
Prod_Desc="hitch"

# 3. Setting Webdriver and Website Specific Information

In [3]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
wait=WebDriverWait(driver, 5)
driver.get('https://dashboard.usaimportdata.com/login.aspx')


In [4]:
email=driver.find_element(By.CSS_SELECTOR, '[name="txtEmail"]')
email.click()
email.send_keys('Vikram.Vadhirajan@firstbrandsgroup.com')
sleep(1)
passw=driver.find_element(By.CSS_SELECTOR, '[name="txtpass"]')
passw.click()
passw.send_keys('FBG@Oct-23')
sleep(1)
driver.find_element(By.CSS_SELECTOR, '[type="submit"]').click()

# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [40]:
cols1 =['Date'] #,'Review_Mentions'
df1= pd.DataFrame(columns=cols1)
count=0

In [7]:
int(driver.find_element(By.CSS_SELECTOR,'[class="sum-results"]').text.split(": ")[1].split(" ")[0])

175

In [36]:
totalResults=int(driver.find_element(By.CSS_SELECTOR,'[class="sum-results"]').text.split(": ")[1].split(" ")[0])
if totalResults <50:
    rpp=50
elif totalResults<100:
    rpp=100
elif totalResults<500:
    rpp=500
elif totalResults <1000:
    rpp=1000

print(rpp)

resperpage=driver.find_element(By.ID, "ddlPageSize")

resperpage.clear()
resperpage.send_keys(rpp)
FirstP=driver.find_element(By.CSS_SELECTOR,'[ng-click="selectPage(1, $event)"]')
FirstP.click()



500


In [45]:
for j in range(2,8):

    Year=Select(driver.find_element(By.CSS_SELECTOR,'[name="ddyear"]'))
    Year.select_by_index(j)

    StartM=Select(driver.find_element(By.CSS_SELECTOR,'[name="ddlfrom"]'))
    StartM.select_by_index(1)

    EndM=Select(driver.find_element(By.CSS_SELECTOR,'[name="ddlto"]'))
    EndM.select_by_index(12)

    prod=driver.find_element(By.CSS_SELECTOR,'[name="txtproduct"]')
    prod.click()
    prod.clear()
    prod.send_keys(Prod_Desc)
    prod.submit()

    HSNC=driver.find_element(By.CSS_SELECTOR,'[name="txtHscode"]')
    HSNC.click()
    HSNC.clear()
    HSNC.send_keys(HSN)
    HSNC.submit()

    Submitkey=driver.find_element(By.CSS_SELECTOR,'[name="btnsearch"]')
    Submitkey.click()

    #____________________________________________________________________________________________________________
    sleep(2)
    input("press enter after modifying the entry")

    #____________________________________________________________________________________________________________

    Dates = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Date"]')
    Consignee_Names = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Consignee_Name"]')
    Shipper_Names = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Shipper_Name"]')
    HSs = driver.find_elements(By.CSS_SELECTOR, '[itemprop=" Hs Code "]')
    Product_Descriptions = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Product_Description"]')
    Weight_in_KGs = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Weight_in_KG"]')
    Quantitys = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Quantity"]')
    Quantity_Units = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Quantity_Unit"]')
    CIFs = driver.find_elements(By.CSS_SELECTOR, '[itemprop="CIF"]')
    Countrys = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Country"]')
    Unloading_Ports = driver.find_elements(By.CSS_SELECTOR, '[itemprop="Unloading_Port"]')

    for Date,Consignee_Name,Shipper_Name,HS,Product_Description,Weight_in_KG,Quantity,Quantity_Unit,CIF,Country,Unloading_Port in zip(Dates,Consignee_Names,Shipper_Names,HSs,Product_Descriptions,Weight_in_KGs,Quantitys,Quantity_Units,CIFs,Countrys,Unloading_Ports):
        df1.loc[count,'Date']=Date.text
        df1.loc[count,'Importer']=Consignee_Name.text
        df1.loc[count,'Exporter']=Shipper_Name.text
        df1.loc[count,'HS Code']=int(HS.text)
        df1.loc[count,'Product Description']=Product_Description.text
        df1.loc[count,'Weight_in_KG']=float(Weight_in_KG.text)
        df1.loc[count,'Quantity']=float(Quantity.text)
        df1.loc[count,'Quantity_Unit']=Quantity_Unit.text
        df1.loc[count,'CIF']=int(CIF.text)
        df1.loc[count,'Export_Country']=Country.text
        df1.loc[count,'Importing_Ports']=Unloading_Port.text
        count=count+1

In [46]:
df1.shape

(791, 11)

In [47]:
df1['Date']=df1['Date'].str.replace("010","10")
df1['Date']=df1['Date'].str.replace("011","11")
df1['Date']=df1['Date'].str.replace("00","0")

In [48]:
df1['year'] = pd.to_datetime(df1['Date']).dt.year


# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [49]:
OFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\11_Adjustable_Trailer_Mount'

In [50]:
with pd.ExcelWriter(OFolder+'\\'+f'ImportData_{Prod_Desc}_{HSN}.xlsx') as writer:  # doctest: +SKIP
    df1.to_excel(writer,index=False, sheet_name='Raw')

# 99. Archived Codes

In [69]:
total=int(driver.find_element(By.CSS_SELECTOR,'[class="col-md-3 "]').text.split(": ")[1].split(" ")[2])

for i in range(total):
    past=int(driver.find_element(By.CSS_SELECTOR,'[class="col-md-3 "]').text.split(": ")[1].split(" ")[0])
    if past<total:
        sleep(3)
        NextP=driver.find_element(By.CSS_SELECTOR,'[class="pagination-next ng-scope"]')
        NextP.click()
    else:
        break